In [ ]:
import os, glob, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.model_selection import GroupKFold
print("cuda:",torch.cuda.is_available(),torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",flush=True)
DEV="cuda" if torch.cuda.is_available() else "cpu"
if DEV=="cuda":
    try:
        _t=torch.nn.Conv1d(2,2,3).to("cuda"); _=_t(torch.zeros(1,2,8,device="cuda")); del _t
        print("GPU conv OK",flush=True)
    except Exception as _e:
        print("GPU unusable, fallback CPU:",_e,flush=True); DEV="cpu"

def find_root():
    for r in ["/kaggle/input/rogii-wellbore-geology-prediction",
              "/kaggle/input/competitions/rogii-wellbore-geology-prediction","data"]+sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(r,"train","*__horizontal_well.csv")): return r
    raise FileNotFoundError("data")
DATA=find_root(); print("DATA:",DATA,flush=True)
STRIDE=2; SCALE=20.0
OFFS=[-40,-25,-15,-8,-4,0,4,8,15,25,40]

def build(w):
    try:
        h=pd.read_csv(f"{DATA}/train/{w}__horizontal_well.csv",usecols=["MD","X","Y","Z","GR","TVT","TVT_input"])
        tw=pd.read_csv(f"{DATA}/train/{w}__typewell.csv").dropna(subset=["TVT","GR"]).sort_values("TVT")
    except Exception: return None
    ps=int(h["TVT_input"].notna().sum())
    if ps<40 or ps>=len(h) or len(tw)<10: return None
    tw_tvt=tw["TVT"].to_numpy(float); tw_gr=tw["GR"].to_numpy(float)
    n=len(h); md=h["MD"].to_numpy(float); z=h["Z"].to_numpy(float); x=h["X"].to_numpy(float); y=h["Y"].to_numpy(float)
    tps=float(h["TVT"].iloc[ps-1]); zps=z[ps-1]; mdps=md[ps-1]
    gr=h["GR"].interpolate(limit_direction="both").to_numpy()
    gr_s=pd.Series(gr).rolling(15,center=True,min_periods=1).median().to_numpy()
    med=np.nanmedian(gr_s); iqr=np.subtract(*np.nanpercentile(gr_s,[75,25]))+1e-6
    tvt_in=h["TVT_input"].to_numpy(); known=np.isfinite(tvt_in).astype(np.float32)
    tvt_in_rel=np.where(known>0,np.nan_to_num(tvt_in-tps),0.0)
    dz_dmd=np.gradient(z)/(np.gradient(md)+1e-9)
    pos=(np.arange(n)-ps)/max(1,n-ps)
    chans=[(gr_s-med)/iqr, np.gradient(gr_s)/5.0,
           pd.Series(gr).rolling(31,center=True,min_periods=1).std().fillna(0).to_numpy()/20.0,
           (z-zps)/50.0, dz_dmd*100.0, pos, tvt_in_rel/20.0, known, (md-mdps)/1000.0]
    for o in OFFS:
        chans.append((gr_s-np.interp(tps+o,tw_tvt,tw_gr))/20.0)
    X=np.stack(chans,1).astype(np.float32)
    ycol=(h["TVT"].to_numpy(float)-tps).astype(np.float32)
    post=(np.arange(n)>=ps)
    sl=slice(0,n,STRIDE)
    twh=int(abs(hash((round(float(tw_tvt[0]),1),round(float(tw_tvt[-1]),1),round(float(np.nansum(tw_gr[:200])),0),len(tw_tvt))))%10**9)
    return dict(well=w,X=X[sl],y=ycol[sl],post=post[sl],sup=twh)

wells=[os.path.basename(f).split("__")[0] for f in sorted(glob.glob(f"{DATA}/train/*__horizontal_well.csv"))]
t0=time.time()
from joblib import Parallel, delayed
seqs=[s for s in Parallel(n_jobs=4)(delayed(build)(w) for w in wells) if s is not None]
C=seqs[0]["X"].shape[1]
print("wells",len(seqs),"channels",C,"%.0fs"%(time.time()-t0),flush=True)

class TCNBlock(nn.Module):
    def __init__(self,c,k,dil):
        super().__init__(); pad=(k-1)*dil//2
        self.net=nn.Sequential(nn.Conv1d(c,c,k,padding=pad,dilation=dil),nn.GELU(),nn.Dropout(0.1),
                               nn.Conv1d(c,c,k,padding=pad,dilation=dil),nn.GELU())
    def forward(self,x): return x+self.net(x)
class TCN(nn.Module):
    def __init__(self,c,d=128,dils=(1,2,4,8,16,32,64)):
        super().__init__()
        self.inp=nn.Conv1d(c,d,1)
        self.blocks=nn.ModuleList([TCNBlock(d,5,dl) for dl in dils])
        self.head=nn.Sequential(nn.Conv1d(d,d,1),nn.GELU(),nn.Conv1d(d,1,1))
    def forward(self,x):
        hh=self.inp(x.transpose(1,2))
        for b in self.blocks: hh=b(hh)
        return self.head(hh).squeeze(1)

WIN=1024; BATCH=16; EPOCHS=18
def crops(train_ws,rng):
    cs=[]
    for si,s in enumerate(train_ws):
        L=len(s["y"])
        for _ in range(2):
            st=0 if L<=WIN else rng.randint(0,L-WIN+1); cs.append((si,st))
    rng.shuffle(cs); return cs
def batches(train_ws,cs):
    for i in range(0,len(cs),BATCH):
        ch=cs[i:i+BATCH]; B=len(ch)
        Xb=np.zeros((B,WIN,C),np.float32); Yb=np.zeros((B,WIN),np.float32); Wb=np.zeros((B,WIN),np.float32)
        for j,(si,st) in enumerate(ch):
            s=train_ws[si]; L=len(s["y"]); e=min(st+WIN,L); m=e-st
            Xb[j,:m]=s["X"][st:e]; Yb[j,:m]=s["y"][st:e]; Wb[j,:m]=s["post"][st:e]
        yield (torch.from_numpy(Xb).to(DEV),torch.from_numpy(Yb).to(DEV),torch.from_numpy(Wb).to(DEV))
@torch.no_grad()
def infer(model,s):
    model.eval(); X=torch.from_numpy(s["X"][None]).to(DEV)
    return model(X)[0].cpu().numpy()*SCALE

sups=np.array([s["sup"] for s in seqs])
gkf=GroupKFold(5); oof={}; 
for k,(tr,va) in enumerate(gkf.split(seqs,groups=sups)):
    tw_=[seqs[i] for i in tr]; vw=[seqs[i] for i in va]
    model=TCN(C).to(DEV); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,EPOCHS); rng=np.random.RandomState(0)
    for ep in range(EPOCHS):
        model.train()
        for Xb,Yb,Wb in batches(tw_,crops(tw_,rng)):
            opt.zero_grad(); p=model(Xb)
            wgt=Wb+0.15*(1-Wb); d=p-Yb/SCALE; loss=(wgt*d*d).sum()/(wgt.sum()+1e-6)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),2.0); opt.step()
        sch.step()
    for s in vw: oof[s["well"]]=infer(model,s)
    print("fold",k,"done %.0fs"%(time.time()-t0),flush=True)

# pooled CV on post-PS strided points
e=[]; 
for s in seqs:
    m=s["post"]; e.append((s["y"][m]-oof[s["well"]][m])**2)
print("const pooled %.3f"%np.sqrt(np.mean(np.concatenate([ (s["y"][s["post"]])**2 for s in seqs]))),flush=True)
print("TCN pooled %.3f"%np.sqrt(np.mean(np.concatenate(e))),flush=True)
np.savez("nn_oof.npz",**{s["well"]:oof[s["well"]] for s in seqs})
print("DONE",flush=True)